# Space-Grade Fault Detector — LibreLane, staged flow
Run from `/foss/designs/Space-Grade-Mechanical-Fault-Detector/`.
Three separate config files, run one at a time: `config_lint.yaml` → `config_synth.yaml` → `config_openroad.yaml`.
Each stage is a standalone LibreLane run (re-reads RTL) — this is a staged review process, not a single continuous flow with shared state.

## 1. Environment check + confirm top.v ports

In [ ]:
import os, yaml
print("PDK_ROOT:", os.environ.get("PDK_ROOT"))
print("CWD:", os.getcwd())
with open("../../rtl/top.v") as f:
    print(f.read()[:2000])  # read the module port declaration — fill CLOCK_PORT below with the real name

## 2. Stage 1 — Lint only (`config_lint.yaml`)
Verilator.Lint plus its checker steps. Catches RTL errors before anything else runs.

In [ ]:
from librelane.flows import SequentialFlow
from librelane.steps import Verilator, Checker

class LintFlow(SequentialFlow):
    Steps = [
        Verilator.Lint,
        Checker.LintTimingConstructs,
        Checker.LintErrors,
        Checker.LintWarnings,
    ]

with open("config_lint.yaml") as f:
    lint_config = yaml.safe_load(f)

lint_flow = LintFlow(lint_config, design_dir="../..")
lint_state, lint_steps = lint_flow.start()

## 3. Stage 2 — Synthesis only (`config_synth.yaml`)
Yosys.Synthesis plus its checkers. Fill in CLOCK_PORT in `config_synth.yaml` from cell 1's output before running this.

In [ ]:
from librelane.steps import Yosys

class SynthFlow(SequentialFlow):
    Steps = [
        Yosys.Synthesis,
        Checker.YosysUnmappedCells,
        Checker.YosysSynthChecks,
        Checker.NetlistAssignStatements,
    ]

with open("config_synth.yaml") as f:
    synth_config = yaml.safe_load(f)

synth_flow = SynthFlow(synth_config, design_dir="../..")
synth_state, synth_steps = synth_flow.start()

## 4. Verify TMR flop count before proceeding
Check the Yosys stats report under this run's `reports/synthesis/` for `tmr_reg_bank` —
confirm the flop count is still 3x, not deduplicated. Don't proceed to stage 3 until this is confirmed.

In [ ]:
print(synth_flow.dir)  # run directory for this stage — inspect reports/synthesis/*.stat.rpt inside it

## 5. Stage 3 — Complete OpenROAD flow (`config_openroad.yaml`)
Full RTL-to-GDSII via the built-in `Classic` flow. Uses `constraints.sdc` via `FALLBACK_SDC` for timing closure.
`RUN_LINTER: false` in this config skips re-linting (already done in stage 1); synthesis still runs internally — Classic has no built-in resume-from-netlist without `--last-run`, so this is expected, not wasted work.

In [ ]:
from librelane.flows import Flow

Classic = Flow.factory.get("Classic")

with open("config_openroad.yaml") as f:
    openroad_config = yaml.safe_load(f)

pnr_flow = Classic(openroad_config, design_dir=".")
pnr_state, pnr_steps = pnr_flow.start()

## 6. Post-flow checks
Inspect `reports/signoff/` in this run's directory for STA (setup/hold slack), DRC, and LVS results
before treating `top` as a hardened macro ready for the pad ring / Chip flow stage.

In [ ]:
print(pnr_flow.dir)